# 05 · Real road photos (v2)

Everything so far was scored on *pasted* signs. Does the model also work on signs photographed on real roads?
v2 tests this with **GTSDB**, 900 dashcam photos (1360×800) of German roads with every sign's box labeled.

The trick is a **paired comparison**. For each real sign, v2 makes two scenes:

* **real**: a 128×128 window cut from the road photo around the sign. First the photo is shrunk so the sign's
  longer side is at most 48 px, as in training; it's never enlarged. The window is placed at random so that it
  holds the whole sign and no other labeled sign.
* **hybrid**: *the same sign's pixels*, cut out of the photo and pasted onto a COCO background exactly the way
  val scenes are made.

Row *i* of `real_dev.npz` and row *i* of `hybrid_dev.npz` are the same physical sign. So if the model does worse
on real than on hybrid, the cause is the real *surroundings* (road context, lighting, camera blur), not the
signs themselves.

**You'll see:** how a window is cut, real/hybrid pairs, a live score of both with the paired table, the
recorded report with its overlap subsets, and real-photo mistakes.

**Needs:** `data/processed/{real,hybrid}_dev.npz`, a trained run, and the GTSDB download (section 1 only).
**Writes:** nothing, unless you turn on a flag.

In [ ]:
# ---- Settings: the only cell you need to edit ----
RUN_NAME = "peak_fresh_aug"  # v2 was measured on this run
REBUILD = False  # True: rebuild real_*/hybrid_* npz files and the GTSDB-vs-GTSRB overlap check (minutes)
RUN_FOR_REAL = False  # True: record real_dev/hybrid_dev scores and the dev report to outputs/eval/<run>/
SCORE_TEST = False  # True: score real_test + hybrid_test ONCE for this run (refused if already scored)

In [ ]:
# Setup (the same in every notebook). All paths in this project (configs/, data/, outputs/) are relative
# to the repo root, and the scripts expect to run from there, so move this notebook's working folder up
# to the root first. Safe to run twice.
import os
import sys
from pathlib import Path

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists():  # walk up until we reach the folder holding pyproject.toml
    if root == root.parent:
        raise RuntimeError("can't find the repo root: open this notebook from inside FindTheObject-CNN-Project")
    root = root.parent
os.chdir(root)
if str(root / "notebooks") not in sys.path:
    sys.path.insert(0, str(root / "notebooks"))  # lets us import notebooks/nbhelpers.py

from nbhelpers import NB_OUT, ROOT, find_run, read_json, require, run_script
print("working in", ROOT)

In [ ]:
# Stop here, with a clear message, if a file this notebook reads is missing; then find the run's checkpoint.
require("data/processed/real_dev.npz", "data/processed/hybrid_dev.npz",
        how="python scripts/make_real_dataset.py --config configs/peak_fresh_aug.yaml")
RUN_DIR = find_run(RUN_NAME)
CKPT = (RUN_DIR / "best.pt").relative_to(ROOT)  # relative, as the scripts record it in their JSON
print("checkpoint:", CKPT)

## 1. From road photo to window

GTSDB's official split is photos 0–599 for **dev** (look as often as you like) and 600–899 for **test**
(scored once). Signs closer than the window allows to another labeled sign are skipped, so each window holds
exactly one sign.

In [ ]:
# Show the road photo that real_dev window 0 came from, with every labeled sign boxed (the window's own sign in lime).
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle

from findobject.data.class_names import CLASS_NAMES
from findobject.data.gtsdb import load_photo, read_gtsdb

real_npz = np.load(ROOT / "data/processed/real_dev.npz")  # the raw arrays, including the photo/sign indexes
photos = read_gtsdb(ROOT / "data/raw/gtsdb/FullIJCNN2013")  # reads gt.txt; opens no photo yet
row = 0
photo = photos[int(real_npz["photo_index"][row])]
target = int(real_npz["sign_index"][row])

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), gridspec_kw={"width_ratios": [3, 1]})
axes[0].imshow(load_photo(photo))
for i, sign in enumerate(photo.signs):
    x1, y1, x2, y2 = sign.box
    axes[0].add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, linewidth=1.5,
                                edgecolor="lime" if i == target else "red"))
axes[0].set_title(f"GTSDB photo {photo.index:05d}.ppm, 1360x800, {len(photo.signs)} labeled sign(s)")
axes[1].imshow(real_npz["images"][row])
x1, y1, x2, y2 = real_npz["boxes"][row]
axes[1].add_patch(Rectangle((x1 - 0.5, y1 - 0.5), x2 - x1, y2 - y1, fill=False, edgecolor="lime"))
axes[1].set_title(f"real_dev window {row}: {CLASS_NAMES[int(real_npz['class_ids'][row])]}\n"
                  f"sign {real_npz['native_long_side'][row]} px in the photo")
for ax in axes:
    ax.set_axis_off()
plt.show()

## 2. Real and hybrid, side by side

Top row: real windows. Bottom row: the same signs pasted onto COCO photos.

In [ ]:
# Load both datasets as PyTorch datasets and draw six pairs (row i of each file is the same physical sign).
from findobject.data.dataset import SignScenes

real = SignScenes(ROOT / "data/processed/real_dev.npz")
hybrid = SignScenes(ROOT / "data/processed/hybrid_dev.npz")
print(len(real), "real windows,", len(hybrid), "hybrid scenes")
fig, axes = plt.subplots(2, 6, figsize=(15, 5.4))
for col, i in enumerate(range(0, 600, 100)):
    for r, (name, ds) in enumerate((("real", real), ("hybrid", hybrid))):
        s = ds[i]
        x1, y1, x2, y2 = s.box.tolist()
        axes[r, col].imshow(s.image.permute(1, 2, 0))
        axes[r, col].add_patch(Rectangle((x1 - 0.5, y1 - 0.5), x2 - x1, y2 - y1, fill=False, edgecolor="lime"))
        axes[r, col].set_title(f"{name} {i}: {CLASS_NAMES[int(s.class_id)]}", fontsize=8), axes[r, col].set_axis_off()
plt.show()

## 3. Score both and compare them pair by pair

Scoring each set on its own and comparing two percentages throws information away. Since every sign appears in
both sets, we can sort the 645 signs into four groups: right in both, right only in real, right only in hybrid,
wrong in both. Signs that are right (or wrong) in both say nothing about real vs hybrid. **Only the two "only"
groups matter.** If real and hybrid were equally hard, a sign that's right in just one of them would be equally
likely to be either, like a coin flip. **McNemar's test** asks how surprising the actual split is under that
coin-flip assumption. A large p-value means "no evidence of a difference".

In [ ]:
# Score the model on both sets (~1 s each), then build the paired table on joint accuracy.
import torch
from torch.utils.data import DataLoader

from findobject.eval.evaluate import evaluate
from findobject.training.checkpoint import load_checkpoint_info, load_model
from findobject.utils.device import pick_device

device = pick_device("auto")
weight = load_checkpoint_info(CKPT).config["train"]["heatmap_loss_weight"]
model = load_model(CKPT, device)
res = {name: evaluate(model, DataLoader(ds, batch_size=256), device, weight)
       for name, ds in (("real", real), ("hybrid", hybrid))}
ok = {name: (r.y_true == r.y_pred) & r.hit for name, r in res.items()}  # joint outcome per sign
for name, r in res.items():
    print(f"{name:6}: joint {r.joint_acc:.4f}  class {r.class_acc:.4f}  pointing {r.pointing_acc:.4f}")

both, only_real = int((ok["real"] & ok["hybrid"]).sum()), int((ok["real"] & ~ok["hybrid"]).sum())
only_hybrid, neither = int((~ok["real"] & ok["hybrid"]).sum()), int((~ok["real"] & ~ok["hybrid"]).sum())
print(f"\n                 hybrid right   hybrid wrong\nreal right       {both:12}   {only_real:12}\n"
      f"real wrong       {only_hybrid:12}   {neither:12}")

In [ ]:
# McNemar's exact test: with n = only_real + only_hybrid coin flips, how likely is a split at least this lopsided?
from math import comb

n, k = only_real + only_hybrid, min(only_real, only_hybrid)
p = min(1.0, 2 * sum(comb(n, i) for i in range(k + 1)) / 2 ** n)  # two-sided: either direction counts
print(f"{only_real} only-real vs {only_hybrid} only-hybrid out of {n} disagreements -> p = {p:.3f}")

## 4. The recorded report and the "seen before?" question

GTSRB (where the training signs come from) and GTSDB were filmed by the same project. So some real windows
might show a sign **the model trained on**, maybe even the very same video frame. A high score on those would
be memory, not generalization. `check_gtsdb_overlap.py` looks for that in two ways, and `report_real.py`
splits the score accordingly:

* **seen**: the exact same frame appears pixel for pixel in GTSRB's training set.
* **unseen**: no exact frame match.
* **unflagged**: no exact match *and* not even suspiciously similar to any training sign. This is the
  cleanest test of generalization.

Each rate comes with a 95% interval (Wilson): with only a few hundred signs, a score is only known to within a
few points.

In [ ]:
# Read the recorded dev report (made by report_real.py) and print each subset with its interval.
report_path = ROOT / "outputs/eval" / RUN_DIR.name / "real_dev_report.json"
if not report_path.exists():
    print(f"{report_path.relative_to(ROOT)} doesn't exist yet: set RUN_FOR_REAL = True and run the last section")
else:
    rep = read_json(report_path)
    print(f"{'subset':10} {'n':>4}   {'real joint (95% interval)':28} {'hybrid joint':12}  McNemar p")
    for name, sub in rep["subsets"].items():
        r, h = sub["real"]["joint"], sub["hybrid"]["joint"]
        print(f"{name:10} {sub['n']:4}   {r['value']:.3f} ({r['lo']:.3f}-{r['hi']:.3f})          {h['value']:.3f}"
              f"         {sub['mcnemar_real_vs_hybrid']['p']:.2f}")

The model does about as well on real photos as on the same signs pasted onto COCO, and no subset shows a
significant gap. Its score on **unseen** signs (0.971) is within a point of all signs (0.975), so memorized frames
aren't what holds the number up.

## 5. Real-photo mistakes

In [ ]:
# Draw every real window the model got wrong: heatmap, peak ring (cyan), true box (green), and both class names.
from findobject.inference.predict import predict
from findobject.viz.overlay import draw_prediction

wrong = torch.nonzero(~ok["real"]).flatten().tolist()
print(len(wrong), "of", len(real), "real windows wrong")
scenes = [real[i] for i in wrong[:16]]
preds = predict(model, torch.stack([s.image for s in scenes]).to(device))
cols = 8
fig, axes = plt.subplots((len(scenes) + cols - 1) // cols, cols, figsize=(17, 2.4 * ((len(scenes) + cols - 1) // cols)),
                         squeeze=False)
for ax in axes.flat:
    ax.set_axis_off()
for ax, s, pred in zip(axes.flat, scenes, preds):
    draw_prediction(ax, s.image, pred, true_box=tuple(s.box.tolist()), true_class=int(s.class_id))
plt.tight_layout()
plt.show()

## Run it for real

**Rebuild the data** (`REBUILD`):

* `make_real_dataset.py` cuts the windows and makes the hybrid pairs, writing `{real,hybrid}_{dev,test}.npz`.
  It's seeded, so you get identical files.
* `check_gtsdb_overlap.py` repeats the seen/unseen search. It writes `outputs/eval/gtsdb_overlap/overlap.json`.
  The reports read the committed copy, `results/v2-gtsdb-overlap/overlap.json`; pass `--overlap` to use a new one.

**Record dev results** (`RUN_FOR_REAL`): score real_dev and hybrid_dev, then build the report from section 4.

**Score test, once** (`SCORE_TEST`): `real_test` and `hybrid_test` count as **one** test family: scoring one
counts as having scored both. So the hybrid call needs `--force`. That's safe here: if real_test was already
scored, the first call is refused and raises, so the forced call never runs.

In [ ]:
# Rebuild the v2 datasets, then the overlap check. Only runs when REBUILD = True.
run_script("make_real_dataset", "--config", "configs/peak_fresh_aug.yaml", enabled=REBUILD)
run_script("check_gtsdb_overlap", "--config", "configs/peak_fresh_aug.yaml", enabled=REBUILD)

In [ ]:
# Score real_dev and hybrid_dev, then write the dev report. Only runs when RUN_FOR_REAL = True.
run_script("evaluate", "--checkpoint", CKPT, "--split", "real_dev", enabled=RUN_FOR_REAL)
run_script("evaluate", "--checkpoint", CKPT, "--split", "hybrid_dev", enabled=RUN_FOR_REAL)
run_script("report_real", "--run", RUN_DIR.name, "--split", "dev", enabled=RUN_FOR_REAL)

In [ ]:
# Score real_test, then hybrid_test (same test family, hence --force), then the test report. Only when SCORE_TEST = True.
run_script("evaluate", "--checkpoint", CKPT, "--split", "real_test", enabled=SCORE_TEST)
run_script("evaluate", "--checkpoint", CKPT, "--split", "hybrid_test", "--force", enabled=SCORE_TEST)
run_script("report_real", "--run", RUN_DIR.name, "--split", "test", enabled=SCORE_TEST)

**Next:** `06_presence_empty_decoys.ipynb`: so far every scene had a sign. What does the model do when there is
none?